# Proyecto Integrador M1

InsightReach es una empresa de marketing digital especializada en campañas personalizadas para negocios locales. Con el crecimiento de su base de clientes y la expansión a nuevos mercados, la empresa busca optimizar su estrategia de segmentación para mejorar la efectividad de sus campañas.

Para ello, se ha diseñado un reto técnico dirigido a candidatos para el rol de Científico de Datos Junior, cuyo objetivo es demostrar su capacidad para integrar múltiples fuentes de datos, analizarlas y generar insights accionables.

## Avance 1

### Cargar y explorar la base de datos de los clientes

In [ ]:
#Primero importamos las librerías necesarias
import numpy as np
import pandas as pd

In [ ]:
#Cargar una base de datos de clientes
url = "https://raw.githubusercontent.com/pachecobravo/ProyectoM1_Christian_Pacheco/refs/heads/main/Documentaci%C3%B3n/base_datos_restaurantes_USA_v2.csv"
df_restaurantes = pd.read_csv(url)

### Exploración inicial

In [ ]:
#Aquí vemos las primeras 5 filas del dataset y cómo están organizados los registros.
print(df_restaurantes.head())

In [ ]:
#Ver el tamaño de la base de datos
df_restaurantes.shape
#Tenemos 30000 filas y 17 columnas, por lo que es una base de datos grande.

In [ ]:
#Veremos el tamaño, tipos de datos y posibles problemas.
df_restaurantes.info()
#Detectamos que existen valores nulos en algunas columnas, 
#por lo que debemos tratarlos antes de continuar con el análisis.
#edad: 101 nulos
#promedio_gasto_comida: 145
#preferencias_alimenticias: 1.403
#telefono_contacto: 15.166
#correo_electronico: 15.072

In [ ]:
#Vemos los nombres de las columnas para tener una idea de qué información tenemos.
df_restaurantes.columns
#Tenemos 17 columnas, que son las siguientes:

In [ ]:
#Estadísticas descriptivas de las columnas numéricas
df_restaurantes.describe()
#Aqui vemos dos variables que tienen valores negativos, que son edad y frecuencia_visita, por lo que debemos tratarlas antes de continuar con el análisis.

#Además vemos el valor máxmimo de edad que es 300, lo cual es un valor muy alto y poco probable, por lo que debemos tratarlo antes de continuar con el análisis.

In [ ]:
#Verificamos los valores negativos en la columna edad
df_restaurantes[df_restaurantes['edad'] < 0]
#Tenemos 97 valores negativos en la columna edad.

In [ ]:
#Verificamos los valores que superan los 100 años en la columna edad
df_restaurantes[df_restaurantes['edad'] > 100]
#Tenemos 110 valores que superan los 100 años en la columna edad, lo cual es un valor muy alto y poco probable.

In [ ]:
#Vemos los tipos de datos de cada columna para tener una idea de qué información tenemos.
df_restaurantes.dtypes
#Aquí llama la atención la edad que es de tipo float, por ahora lo dejamos así

##### Observaciones de la exploración inicial
- La base tiene 30.000 filas y 17 columnas (3 numéricas (int), 2 decimales de tipo float, 12 de texto(str)).
- `id_persona` es un identificador; sus estadísticas no tienen interpretación.
- En `edad` el mínimo es -5 y el máximo 300, y en `frecuencia_visita` el mínimo es -3. Son valores
  que parecen imposibles y se revisarán en la limpieza.
- `ingresos_mensuales` va de 800 a 17.999 y `promedio_gasto_comida` de 0 a 149,97.
- `info()` muestra columnas con menos de 30.000 valores no nulos; se tratarán en la limpieza.

### Limpieza de datos, identificar valores nulos, duplicados y errores.

In [ ]:
#Vemos la cantidad de valores nulos en cada columna para tener una idea de qué información tenemos.
df_restaurantes.isnull().sum()
#Tenemos 5 columnas con valores nulos, que son las siguientes:
#edad: 101 nulos
#promedio_gasto_comida: 145 nulos
#preferencias_alimenticias: 1403 nulos
#telefono_contacto: 15166 nulos
#correo_electronico: 15072 nulos

In [255]:
#Verificamos si existen valores duplicados en el dataframe
df_restaurantes.duplicated().sum()
#No hay filas duplicadas

np.int64(0)

In [ ]:
#Esta columna al tener valores atípicos, usamos la mediana.
#Reemplazamos los valores outliers en la columna edad por nan, para poder tratarlos después.
df_restaurantes['edad'] = df_restaurantes['edad'].replace([-5, 300], np.nan)

In [ ]:
#Antes de rellenar debemos calcular la edad mediana por género, para poder rellenar los valores nulos de manera más precisa.
df_restaurantes.groupby('genero')['edad'].median()
#Tenemos la misma mediana en ambos géneros

In [ ]:
#Ahora rellenamos los valores nulos de la columna edad con la mediana de cada género.
df_restaurantes['edad'] = df_restaurantes.groupby('genero')['edad'].transform(
    lambda x: x.fillna(x.median())
)

#Verificamos si quedó bien la limpieza
df_restaurantes['edad'].describe()
#Ya no existe valores erróneos e la columna edad

In [ ]:
#Ahora vamos a verificar cual es la mediana en la columna de frecuencia_visita, de acuedo a cada género
df_restaurantes[df_restaurantes['frecuencia_visita'] >= 0].groupby('genero')['frecuencia_visita'].median()
#Vemos que es la misma para cada uno de 4

In [ ]:
#Ahora reemplazamos los valores negativos en la columna frecuencia_visita por la mediana de cada género, que es 4.
df_restaurantes['frecuencia_visita'] = df_restaurantes[
    'frecuencia_visita'
].replace(-3, 4)

#Verificamos los los valores menores a 0
df_restaurantes['frecuencia_visita'].value_counts().sort_index()
#Vemos que no existen valores negativos.

In [ ]:
#Ahora vamos con la siguiente columna que es promedio_gasto_comida, por los valores nulos que contiene.
df_restaurantes['promedio_gasto_comida'].isna().sum()
#Tenemos 145 valores nulos que los vamos a reemplazar por la mediana de cada género, para poder rellenar los valores nulos de manera más precisa.

In [ ]:
#Verificamos las medianas de cada género en la columna promedio_gasto_comida
df_restaurantes.groupby('genero')['promedio_gasto_comida'].median()
#Vemos que es similar en cada grupo de género.

In [ ]:
#Realizamos la limpieza de la columna promedio_gasto_comida, reemplazando los valores nulos por la mediana de cada género.
df_restaurantes['promedio_gasto_comida'] = df_restaurantes.groupby('genero')['promedio_gasto_comida'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
#Verificamos si quedó bien la limpieza
df_restaurantes['promedio_gasto_comida'].isna().sum()
#Tenemos 0 valores nulos en la columna promedio_gasto_comida, por lo que la limpieza fue exitosa.

In [ ]:
#Ahora verificamos la columna preferencias_alimenticias.
df_restaurantes['preferencias_alimenticias'].value_counts(dropna=False)
#Vemos que tiene 1.403 valores nulos.

In [ ]:
#Sustituimos missing values por "No especificado" en la columna preferencias_alimenticias.
df_restaurantes['preferencias_alimenticias'] = df_restaurantes[
    'preferencias_alimenticias'
].fillna('No especificado')
#Este cambio de nombre se lo hace por que pueden haber más preferencias de los clientes que no estén en la lista

In [ ]:
#Verificamos si quedó bien la limpieza
df_restaurantes['preferencias_alimenticias'].value_counts()
#Tenemos los valores reemplazados correctamente.

In [ ]:
df_restaurantes.isna().sum()
#Ya no hay nulos en las columnas de comportamiento. Teléfono y correo se dejan sin tratar: son datos de contacto.

### Filtrar la base de datos para una ciudad específica.

In [ ]:
#Revisamos el numero de datos de cada ciudad
df_restaurantes['ciudad_residencia'].value_counts()
#Elegimos trabajar con los datos de Chicago

In [254]:
#Creamos un nuevo Data Frame
df_chicago = df_restaurantes[df_restaurantes['ciudad_residencia'] == 'Chicago'].copy()

In [ ]:
#Verificamos numero de filas y columnas
df_chicago.shape
#Cuenta con 5384 filas y 17 columnas, por lo que es una base de datos bastante grande para trabajar.

### Tabulación

#### Tablas básicas para entender la distribución de variables.

##### Columnas categóricas

In [ ]:
#Ahora vemos el porcentaje de cada género en la base de datos.
df_chicago['genero'].value_counts(normalize=True) * 100
#No existe mucha diferencia entre los géneros, por lo que podemos decir que la base de datos está balanceada en cuanto a género.

In [ ]:
#Ahora queremos saber los porcentajesd de las preferencias alimenticias sobresalen en el data frame de Chicago.
df_chicago['preferencias_alimenticias'].value_counts(normalize=True) * 100
#Las preferencias de carnes y vegetariano sobresalen con un 29% y 20% respectivamente.

In [ ]:
#Vemos los porcentajes en la columna de consumo de licor
df_chicago['consume_licor'].value_counts(normalize=True) * 100
#Un 62% de los clientes consume licor, mientras que un 38% no lo hace.

In [ ]:
#Ahora vemos los porcentajes de la columna membresia_premium y verificamos quienes tienen
df_chicago['membresia_premium'].value_counts(normalize=True) * 100
#El 58% de los clientes no tienen membresía premium, mientras que un 42% si la tiene.

In [ ]:
#Verificamos los porcentajes de la columna tipo_de_pago_mas_usado
df_chicago['tipo_de_pago_mas_usado'].value_counts(normalize=True) * 100
#Vemos que el tipo de pago más usado es en efectivo con un 39%, seguido por tarjeta con 33% y app con 25%.

In [ ]:
#Pesos porcentuales
df_chicago['estrato_socioeconomico'].value_counts(normalize=True) * 100
#Vemos que el 33% de los clientes pertenecen al estrato socioeconómico medio, seguido por el 28% que pertenecen al estrato alto.

In [ ]:
#Pesos porcentuales
df_chicago['ocio'].value_counts(normalize = True) * 100
#Vemos que no hay tanta diferencia entre los que tienen ocio y los que no

#### Columnas Numéricas

In [ ]:
#Analizamos la columna edad para ver la distribución de edades de los clientes.
df_chicago['edad'].describe()

In [ ]:
#Separamos las edades por rangos y creamos una nueva columna llamada grupo_edad.
df_chicago['grupo_edad'] = pd.cut(
    df_chicago['edad'],
    bins=[17, 30, 50, 65, 80],
    labels=['18-30', '31-50', '51-65', '66-80']
)
#Vemos como quedo la columna grupo_edad
df_chicago['grupo_edad'].value_counts()
#El rango predominante es el de los 31 a 50 años, seguido por el de 66 a 80 años.

In [ ]:
#Analizamos la columna frecuencia_visita para ver la distribución de visitas de los clientes.
df_chicago['frecuencia_visita'].describe()

In [ ]:
#Aquí vemos como se distribuyen las visitas de los clientes.
df_chicago['frecuencia_visita'].value_counts()
#La mayor cantidad de clientes visitan el restaurante 4 veces al mes, seguido por los que visitan 3 veces al mes y 5 veces al mes.

In [ ]:
#Separamos en 4 rangos la cantidad de visitas
df_chicago['grupo_frecuencia'] = pd.cut(
    df_chicago['frecuencia_visita'],
    bins=[-1, 2, 5, 7, 10],
    labels=['Baja', 'Media', 'Alta', 'Muy alta']
)

#Revisamos como quedo la columna grupo_frecuencia
df_chicago['grupo_frecuencia'].value_counts()
#Vemos que la mayoría de los clientes tienen una frecuencia de visita media, seguido por los que tienen una frecuencia baja y alta.

In [ ]:
#Analizamos la columna del promedio del gasto en comida.
df_chicago['promedio_gasto_comida'].describe()

In [ ]:
#Separamos en 4 rangos el promedio del gasto en comida y creamos una nueva columna llamada grupo_gasto.
df_chicago['grupo_gasto'] = pd.cut(
    df_chicago['promedio_gasto_comida'],
    bins=[-1, 25, 50, 80, 105],
    labels=['Bajo', 'Medio', 'Alto', 'Muy alto']
)

#Revisamos como quedo la columna grupo_gasto
df_chicago['grupo_gasto'].value_counts()
#1.463 registros tienen 0 de gasto y coinciden con frecuencia de visita 0 por lo que son válidos.

In [ ]:
#Revisamos la columna de ingresos mensuales
df_chicago['ingresos_mensuales'].describe()

In [ ]:
#Dividimos en 4 rangos la columna de ingresos mensuales y creamos una nueva columna llamada grupo_ingresos.
df_chicago['grupo_ingresos'] = pd.cut(
    df_chicago['ingresos_mensuales'],
    bins=[0, 4500, 9000, 13500, 18000],
    labels=['Bajo', 'Medio', 'Alto', 'Muy alto']
)

#Revisamos como quedo la columna grupo_ingresos
df_chicago['grupo_ingresos'].value_counts().sort_index()
#Vemos que la mayoría de los clientes tienen ingresos mensuales bajos, seguido por los que tienen ingresos medios y altos.

In [ ]:
df_chicago.head()

### Conclusiones del Avance 1

- Se exploró y limpió la base de 30.000 clientes: se corrigieron edades y frecuencias de visita
  imposibles, se imputaron los nulos con la mediana por género y no se encontraron duplicados.
- Teléfono y correo se dejaron sin tratar: son datos de contacto y no describen el comportamiento.
- Se filtró la ciudad de **Chicago** (5.384 clientes), la que más registros tiene.
- Los clientes de Chicago se reparten de forma equilibrada entre géneros, prefieren carnes (29,5%) y
  platos vegetarianos (20,9%), y el 62,7% consume licor.
- Solo el 41,1% tiene membresía premium, por lo que hay una base amplia de clientes que podría
  convertirse a premium.
- La mayoría tiene gasto bajo (61% hasta 25) e ingresos bajos (60% hasta 4.500), con una frecuencia
  típica de 4 visitas al mes.